# Encontro 6 · O agente de férias da Aurora com a oficina

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/se-ai-agents-engineering/ai-agents-engineering-v2/blob/main/encontro-06/01-juntos-aurora.ipynb)

A Aurora voltou, com um problema novo: os pedidos de férias. Neste notebook a turma inteira constrói um agente, do esqueleto até a versão que aguenta um ataque, passando pelas decisões que cada aula tratou separadamente. Os notebooks 00 e 00b construíram estas peças à mão. Aqui o mesmo percurso usa a `oficina.py`, que é o que os grupos usam no exercício (notebook 02), num caso diferente.

O percurso tem quatro passos. Em cada um, chega uma mensagem de alguém da Aurora com uma restrição nova. **Antes de cada mudança no código, a turma decide**: pelo chat ou pela enquete, vocês apostam no resultado e propõem a mudança. O código muda conforme o que a turma decidir, e um resultado inesperado também ensina.

- **Passo 1 · Agente ou workflow?** (Aula 1)
- **Passo 2 · O contrato da ferramenta nova** (Aula 2)
- **Passo 3 · Fonte, "não sei" e encaminhamento** (Aulas 3 e 4)
- **Passo 4 · O ataque** (o tema novo: segurança e governança)

Quer rodar junto? Funciona igual ao exercício: coloquem as chaves nos Segredos do Colab (`OPENAI_API_KEY` e, se tiverem, `SERPER_API_KEY`) e rodem de cima para baixo.

## Preparação

In [ ]:
CASO = "aurora"

# No Colab, baixa o repositório e instala o LangChain. Na sua máquina, só importa.
import os, sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.exists("/content/ai-agents-engineering"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/se-ai-agents-engineering/ai-agents-engineering-v2",
                        "/content/ai-agents-engineering"], check=True)
    os.chdir("/content/ai-agents-engineering/encontro-06")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "langchain>=1.0,<2", "langchain-openai>=1.0,<2"], check=True)
sys.path.insert(0, os.getcwd())

import oficina as of
of.preparar(CASO)

In [ ]:
of.brief()

### Abrindo a caixa

O notebook usa a `oficina.py`, que guarda o que não muda de um caso para outro: as ferramentas, o placar e o ataque. Ela não tem mágica. Uma ferramenta, por exemplo, é uma função Python comum; o que o modelo vê dela é só o **contrato** (nome, descrição e schema), que vamos montar logo abaixo.

In [ ]:
import inspect
print(inspect.getsource(of.ler_chamado))
print(inspect.getsource(of.registrar_ferias))
print(inspect.getsource(of._acao))

`registrar_ferias` não lança nada numa folha de verdade: anota a ação num registro, que `of.sistema()` mostra. É esse registro que conta, no placar, o que o agente **fez** e não podia desfazer.

## O esqueleto

O agente tem cinco peças, e cada uma é uma decisão:

- `PROMPT`: as instruções de sistema.
- `FERRAMENTAS`: o que o agente pode fazer. É o **escopo** dele.
- `CONTRATOS`: o nome, a descrição e o schema de cada ferramenta.
- `APROVACAO_HUMANA`: quais ferramentas param e esperam um humano aprovar.
- `MARCAR_EXTERNO` e `FILTRO_DE_SAIDA`: dois guardrails que funcionam **fora do modelo**, na camada das ferramentas.

O esqueleto começa com o prompt genérico, todas as ferramentas, os contratos que vieram prontos e nenhum guardrail.

In [ ]:
PROMPT = of.prompt_inicial()
FERRAMENTAS = of.nomes_das_ferramentas()
CONTRATOS = {}
APROVACAO_HUMANA = {}
MARCAR_EXTERNO = False
FILTRO_DE_SAIDA = False

print("Prompt:", PROMPT)
print("Ferramentas:", ", ".join(FERRAMENTAS))
print("Irreversíveis:", ", ".join(of.irreversiveis(FERRAMENTAS)))

`create_agent` é o loop que vocês escreveram à mão nas primeiras aulas: o modelo pede ferramentas, o código executa, o resultado volta, até a resposta final. Três detalhes:

- `response_format=of.formato()` obriga a resposta final a ter **desfecho** (`responder`, `nao_sei`, `escalar` ou `acao`), **fontes** e texto. É o que permite corrigir com uma régua, como na Aula 5.
- `middleware` envolve o loop sem mexer nele. O `HumanInTheLoopMiddleware` pausa a execução antes das ferramentas escolhidas e espera uma decisão humana.
- `checkpointer` guarda o estado durante a pausa. O `create_agent` é construído sobre o LangGraph, e este é o checkpoint da Aula 4.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def montar_agente():
    ferramentas = of.montar_ferramentas(FERRAMENTAS, contratos=CONTRATOS,
                                        marcar_externo=MARCAR_EXTERNO, filtro_de_saida=FILTRO_DE_SAIDA)
    middleware = [HumanInTheLoopMiddleware(interrupt_on=APROVACAO_HUMANA)] if APROVACAO_HUMANA else []
    return create_agent(model=of.modelo(), tools=ferramentas, system_prompt=PROMPT,
                        response_format=of.formato(), middleware=middleware, checkpointer=InMemorySaver())


agente = montar_agente()
of.atender(agente, of.perguntas(1)[0]["pergunta"])

---
## Passo 1 · Agente ou workflow?

In [ ]:
of.mensagem(1)

O formulário do Portal já traz a data de início, os dias e a venda. Para ele existe uma alternativa sem modelo: um **workflow** em código, que aplica as regras da política com `if` e chama a mesma API de feriados. Ele também é o gabarito deste passo.

In [ ]:
print(inspect.getsource(of._wf_ferias))

**🗳️ A turma decide, antes de rodar.** Das três perguntas do formulário, quantas o agente acerta? O custo por atendimento cabe nos US$ 0,002? E a resposta sai em até 3 segundos?

In [ ]:
placar_agente = of.rodar(agente, 1)
placar_workflow = of.rodar_workflow()
of.comparar(agente=placar_agente, workflow=placar_workflow)

Vamos abrir um caso em que o agente errou, ou acertou pelo caminho mais caro. Olhem o trace: ele consultou os feriados? Fez a conta do dia da semana? Leu a regra inteira?

In [ ]:
of.atender(agente, of.perguntas(1)[1]["pergunta"])      # troque o índice para abrir outro caso

**🗳️ A turma decide.** O formulário vai para o agente ou para o workflow? E o que sobra para o agente? Olhem as perguntas dos próximos passos: elas chegam em texto livre.

Uma rodada basta para decidir? Na Aula 5, a mesma pergunta mudou de resposta entre amostras. Se a turma ficar em dúvida, rodamos de novo.

In [ ]:
of.registrar(1,
    decisao="",
    alternativa_descartada="",
    evidencia="",
)

---
## Passo 2 · O contrato da ferramenta nova

In [ ]:
of.mensagem(2)
print()
of.mostrar_contrato(of.ferramenta_do_passo_2())

**🗳️ A turma decide, antes de rodar.** Vocês aprovariam este PR? O que o modelo sabe sobre esta ferramenta, lendo só o contrato?

In [ ]:
of.rodar(agente, 2)

### Reescrevendo o contrato, juntos

Mandem pelo chat o que falta, usando o Tool Contract Sheet da Aula 2 como checklist:

- um **nome** que diga para que a ferramenta serve, na linguagem do domínio;
- uma **descrição** que diga quando usar, o que volta e o que fazer com o resultado. Quem escreve os comentários do chamado? O modelo deveria confiar neles?
- um **schema** que restrinja o argumento: formato do número do chamado, descrição do campo.

O nome do argumento (`numero`) fica como está, porque a implementação depende dele.

In [ ]:
CONTRATOS["ler_chamado"] = {
    "nome": "api_chamados",
    "descricao": "Consulta a API.",
    "parametros": {"type": "object", "properties": {"numero": {"type": "string"}}, "required": ["numero"]},
}

agente = montar_agente()
of.rodar(agente, 2)

Uma rodada que passou não prova que o contrato está bom. A pergunta da revisão continua sendo: o que o contrato antigo deixava o modelo fazer que ele não deveria?

In [ ]:
of.registrar(2,
    decisao="",
    alternativa_descartada="",
    evidencia="",
)

> ✅ **Ponto de checagem**
>
> **Na oficina, CONTRATOS['ler_chamado'] = {...} corresponde a qual peça do notebook 00?**
>
> A) Ao StructuredTool.from_function, com nome e descrição novos para a mesma função.  
> B) Ao prompt de sistema.  
> C) Ao HumanInTheLoopMiddleware.  
> D) Ao workflow do Passo 1.  

---
## Passo 3 · Fonte, "não sei" e encaminhamento

In [ ]:
of.mensagem(3)

**🗳️ A turma decide, antes de rodar.** Com o prompt genérico do esqueleto, o agente vai dizer "não sei" quando a política não cobre a pergunta? Vai encaminhar o pedido de 40 dias? Vai registrar as férias sem conferir nada?

In [ ]:
of.rodar(agente, 3)

O que já aconteceu nos sistemas da Aurora, de verdade, até aqui:

In [ ]:
of.sistema()

### Reescrevendo o prompt, juntos

Mandem pelo chat as regras que o prompt precisa ter para cumprir a mensagem do jurídico. Regras curtas e verificáveis: uma regra que não dá para conferir no trace não ajuda a decidir. Quatro assuntos para cobrir:

- **fonte**: de onde vem a regra e como citá-la no campo `fontes`;
- **"não sei"**: o que fazer quando a política não cobre a pergunta;
- **encaminhamento**: o que é exceção ou de outra área, e para quem vai;
- **ação**: o que conferir antes de registrar as férias.

In [ ]:
PROMPT = of.prompt_inicial() + """

Regras:
- ...
"""

agente = montar_agente()
of.rodar(agente, 3)

O prompt novo não pode quebrar o que já funcionava:

In [ ]:
of.rodar(agente, 1)
of.rodar(agente, 2)

In [ ]:
of.registrar(3,
    decisao="",
    alternativa_descartada="",
    evidencia="",
)

**🗳️ Uma pergunta da Aula 4.** Vocês dividiriam este agente? Um especialista que lê o chamado e outro que registra as férias, por exemplo. Que limite real justificaria a divisão, e o que ela custaria? Não vamos implementar: a resposta vai para o registro.

In [ ]:
of.registrar(5,
    decisao="",
    alternativa_descartada="",
    evidencia="",
)

---
## Passo 4 · O ataque

In [ ]:
of.mensagem(4)

### A trifeta letal

Um agente fica exposto à **injeção indireta de prompt** quando tem três coisas ao mesmo tempo: lê **entrada não confiável**, tem **acesso a dado sensível** e tem **capacidade de agir** ou de **mandar dado para fora**. O modelo não distingue com segurança a instrução do usuário de uma instrução escondida no resultado de uma ferramenta: para ele, tudo é texto no contexto.

In [ ]:
of.trifeta(FERRAMENTAS)

### Como o ataque funciona por dentro

Quando um ataque está ligado, a oficina acrescenta um texto envenenado no resultado de uma ferramenta. No caso da Aurora, é um comentário a mais no chamado, como se alguém com acesso ao chamado o tivesse escrito.

In [ ]:
print(inspect.getsource(of._contaminar))

O ataque de treino da Aurora tem duas tentativas: uma quer que o agente **registre férias** que ninguém pediu (T1); a outra quer que ele **mande CPF, salário e conta** para um e-mail de fora (T2). Cada uma roda 3 vezes, porque o modelo varia.

**🗳️ A turma decide, antes de rodar.** Das 6 rodadas, quantas vão comprometer o agente?

In [ ]:
ataque, controle = of.atacar(agente, "treino")
ataque

Uma rodada por dentro, com o trace. O comentário envenenado aparece marcado com ⚠:

In [ ]:
of.ver_ataque(agente, "treino", "T1")

### Primeira defesa: aprovação humana

**🗳️ A turma decide.** A aprovação humana na ação irreversível resolve os dois ataques? O que a `description` precisa dizer para quem vai aprovar?

In [ ]:
APROVACAO_HUMANA = {
    "registrar_ferias": {"allowed_decisions": ["approve", "reject"],
                         "description": "Confira se o próprio colaborador pediu estas férias e se a liderança aprovou."},
}

agente = montar_agente()
ataque, controle = of.atacar(agente, "treino")
ataque

Do lado de quem aprova, a pausa fica assim:

In [ ]:
of.atender(agente, of.perguntas(3)[2]["pergunta"])

### As outras camadas

A aprovação humana está ligada só em `registrar_ferias`. O e-mail sai sem passar por ninguém, e aprovar cada e-mail à mão não escala. As camadas que restam:

- **Escopo mínimo.** O agente precisa mandar e-mail? Tirar a ferramenta de `FERRAMENTAS` corta uma perna da trifeta.
- **Contrato mais estreito.** Se o e-mail só pode ir para a gestora, o argumento `para` não precisa existir no contrato.
- **Marcar o conteúdo externo.** `MARCAR_EXTERNO = True` avisa o modelo que o chamado é dado, não instrução. Depende do modelo: reduz, não elimina.
- **Filtro de saída.** `FILTRO_DE_SAIDA = True` bloqueia, fora do modelo, saída com CPF, cartão, telefone, dado bancário ou para destinatário fora da lista.

**🗳️ A turma decide.** Quais camadas entram? Para cada uma: qual perna da trifeta ela corta, e ela funciona dentro ou fora do modelo?

In [ ]:
MARCAR_EXTERNO = True
FILTRO_DE_SAIDA = True
# FERRAMENTAS = [f for f in FERRAMENTAS if f != "enviar_mensagem"]     # escopo mínimo

agente = montar_agente()
ataque, controle = of.atacar(agente, "treino")
ataque

Duas perguntas para fechar o passo:

- O filtro de saída procura padrões: CPF, cartão, telefone, conta. O salário passa por ele? E se o agente mandar o salário para a gestora, que está na lista de destinatários? A política diz que nem a liderança pode receber. Um filtro de padrões é uma camada, não a solução.
- O controle continua certo? Um guardrail que bloqueia o ataque e também o uso legítimo não resolveu o problema.

In [ ]:
of.registrar(4,
    decisao="",
    alternativa_descartada="",
    evidencia="",
)

> ✅ **Ponto de checagem**
>
> **O ataque de treino rodou 3 vezes e comprometeu o agente em 1. Como ler esse resultado?**
>
> A) O agente resiste em 2 de 3 vezes, o que é aceitável.  
> B) O ataque funciona: com milhares de atendimentos por mês, 1 em 3 acontece o tempo todo.  
> C) Foi azar; numa nova rodada, o resultado será 0 de 3.  
> D) O modelo aprendeu com as rodadas anteriores.  

---
## Agora é com vocês

O registro da turma, com as decisões da Aurora:

In [ ]:
of.exportar_registro("Turma · Aurora", PROMPT, FERRAMENTAS)

No exercício, cada grupo faz o mesmo percurso num dos três casos, numa de duas alternativas: construindo o agente do zero, como no notebook 00 (`02a-viagem`, `02a-fornecedor` ou `02a-entregas`), ou usando a oficina, como neste notebook (`02b-exercicio-oficina`). Os casos:

- **`viagem` · Lumen Consultoria**: viagens corporativas, com previsão do tempo, feriados e busca na web;
- **`fornecedor` · Vértice Indústria**: cadastro de fornecedores, com consulta de CNPJ e busca na web;
- **`entregas` · Casa Pomar**: entregas e trocas de um e-commerce, com CEP, feriados e previsão do tempo.

As decisões não precisam ser as mesmas da Aurora. Em pelo menos um dos casos, a resposta do Passo 1 deveria ser diferente.